# Smart MCQ Solver - DL & GenAI Project

# 1. Environment Setup

Import all the required libraries that will be used throughout this project.

In [1]:
import os
import re
import json
import warnings

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics.pairwise import cosine_similarity

print("Environment Ready ✅")

Environment Ready ✅


# 2. Configuration

Define project constants and file paths used throughout the notebook.

In [2]:
SEED = 42
VAL_SIZE = 0.2

TRAIN_PATH = "/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"
TEST_PATH = "/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv"
SAMPLE_SUBMISSION_PATH = "/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv"

OPTIONS = ["A", "B", "C", "D", "E"]

# 3. Load the Dataset

Load the training and test datasets and inspect their basic structure before performing any preprocessing or modeling.

In [3]:
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
sample_submission = pd.read_csv(SAMPLE_SUBMISSION_PATH)

print(f"Train Shape: {train.shape}")
print(f"Test Shape: {test.shape}")

train.head()

Train Shape: (2000, 8)
Test Shape: (500, 7)


,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


# 4. Dataset Overview

Understand the structure, data types, and completeness of the dataset.

In [4]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 8 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   id      2000 non-null   int64 
 1   prompt  2000 non-null   object
 2   A       2000 non-null   object
 3   B       2000 non-null   object
 4   C       2000 non-null   object
 5   D       2000 non-null   object
 6   E       2000 non-null   object
 7   answer  2000 non-null   object
dtypes: int64(1), object(7)
memory usage: 125.1+ KB


# 5. NLP 

Machine learning models cannot understand raw text. Before applying any learning algorithm, textual data must be converted into numerical representations. In this milestone, we build progressively better text representations, starting from TF-IDF and Word2Vec.

In [5]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"\s+", " ", text)
    return text.strip()

text_columns = ["prompt", "A", "B", "C", "D", "E"]

for col in text_columns:
    train[col] = train[col].apply(clean_text)
    test[col] = test[col].apply(clean_text)

## Tokenization

Tokenization is the process of breaking raw text into smaller units called tokens. These tokens form the basic input for almost every NLP model, from TF-IDF to modern Transformer architectures.

In [6]:
sample_text = train.loc[0, "prompt"]

tokens = sample_text.split()

print("Original Text:\n")
print(sample_text)

print("\nTokens:\n")
print(tokens)

Original Text:

pick the best possible answer: what is martin heidegger's view on the relationship between time and human existence? among the listed options.

Tokens:

['pick', 'the', 'best', 'possible', 'answer:', 'what', 'is', 'martin', "heidegger's", 'view', 'on', 'the', 'relationship', 'between', 'time', 'and', 'human', 'existence?', 'among', 'the', 'listed', 'options.']


## TF-IDF (Term Frequency - Inverse Document Frequency)

Bag of Words treats every word as equally important. However, common words such as "is", "the", and "of" appear in almost every document and contribute very little information.

TF-IDF improves upon Bag of Words by assigning higher weights to important words while reducing the influence of very common words.

### TF-IDF Vectorization

In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer()

train_corpus = (
    train["prompt"] + " " +
    train["A"] + " " +
    train["B"] + " " +
    train["C"] + " " +
    train["D"] + " " +
    train["E"]
)

tfidf_matrix = tfidf.fit_transform(train_corpus)

print("TF-IDF Matrix Shape:", tfidf_matrix.shape)

TF-IDF Matrix Shape: (2000, 2940)


### Similarity using TF-IDF

Once every document has been converted into TF-IDF vectors, we can compare two pieces of text by measuring how similar their vectors are. Cosine Similarity is the most commonly used metric for this purpose.

In [8]:
sample_prompt = train.loc[0, "prompt"]

option_vectors = tfidf.transform(train.loc[[0], ["A", "B", "C", "D", "E"]].values.flatten())
prompt_vector = tfidf.transform([sample_prompt])

scores = cosine_similarity(prompt_vector, option_vectors).flatten()

for option, score in zip(OPTIONS, scores):
    print(f"{option}: {score:.4f}")

A: 0.2619
B: 0.2953
C: 0.5777
D: 0.5238
E: 0.2332


## Word2Vec Embeddings

Unlike TF-IDF, which represents words using frequency statistics, Word2Vec learns dense vector representations where semantically similar words are placed closer together in the embedding space.

For this project, Word2Vec serves as a conceptual improvement over TF-IDF before moving to Transformer-based embeddings.

In [9]:
from gensim.models import Word2Vec

sentences = [text.split() for text in train["prompt"]]

word2vec = Word2Vec(
    sentences,
    vector_size=100,
    window=5,
    min_count=1,
    workers=4,
    seed=SEED
)

print(word2vec.wv.most_similar("following", topn=5))

[('correct?', 0.9953771233558655), ('accurately', 0.9910875558853149), ('which', 0.9903026223182678), ('statements', 0.9859045743942261), ('of', 0.983677089214325)]


## Evaluation Metric - Mean Average Precision @ 3 (MAP@3)

The competition is evaluated using MAP@3 instead of accuracy. Since each prediction consists of the top three ranked answer choices, this metric rewards models that rank the correct answer higher.

In [10]:
def average_precision_at_3(actual, predicted):
    if actual in predicted[:3]:
        return 1 / (predicted[:3].index(actual) + 1)
    return 0

def map_at_3(actuals, predictions):
    scores = [
        average_precision_at_3(a, p)
        for a, p in zip(actuals, predictions)
    ]
    return np.mean(scores)

# 6. Transformer-based NLP

Classical embedding methods such as TF-IDF and Word2Vec have significant limitations in understanding context and semantics. Transformer models overcome these limitations by learning contextual representations of language through the attention mechanism.

## Context-Aware Sentence Embeddings

Sentence Transformers convert complete sentences into dense vector representations while preserving semantic meaning. These embeddings can be compared using cosine similarity for semantic retrieval and ranking.

In [11]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

sample_sentences = train.loc[:4, "prompt"].tolist()

embeddings = embedding_model.encode(sample_sentences)

print("Embedding Shape:", embeddings.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding Shape: (5, 384)


### Semantic Similarity using Sentence Transformers

Sentence embeddings capture the semantic meaning of text. By comparing the embedding of a question with the embeddings of its answer choices, we can rank the options based on semantic similarity.

In [12]:
sample = train.iloc[0]

texts = [sample["prompt"]] + [sample[option] for option in OPTIONS]

embeddings = embedding_model.encode(texts)

prompt_embedding = embeddings[0]
option_embeddings = embeddings[1:]

scores = cosine_similarity(
    [prompt_embedding],
    option_embeddings
).flatten()

for option, score in zip(OPTIONS, scores):
    print(f"{option}: {score:.4f}")

A: 0.7308
B: 0.7658
C: 0.7940
D: 0.7701
E: 0.7323


### Zero-shot Classification

Zero-shot classification uses a pretrained Natural Language Inference (NLI) model to determine how well each answer option matches the given question, without requiring any additional training on the dataset.

Each answer choice is scored independently, and the highest scoring options are selected as the predicted answers.

In [13]:
from transformers import pipeline

zero_shot = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli"
)

sample = train.iloc[0]

result = zero_shot(
    sample["prompt"],
    candidate_labels=[sample[option] for option in OPTIONS],
    multi_label=False
)

for label, score in zip(result["labels"], result["scores"]):
    print(f"{score:.4f}  {label}")

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

0.4777  martin heidegger believes that humans do not exist inside time, but that they are time. the relationship to the past is a present awareness of having been, and the relationship to the future involves anticipating a potential possibility, task, or engagement.
0.2054  martin heidegger believes that humans exist within a time continuum that is infinite and does not have a defined beginning or end. the relationship to the past involves acknowledging it as a historical era, and the relationship to the future involves creating a world that will endure beyond one's own time.
0.1274  martin heidegger believes that the relationship between time and human existence is cyclical. the past and present are interconnected and the future is predetermined. human beings do not have free will.
0.1134  martin heidegger does not believe in the existence of time or that it has any effect on human consciousness. the relationship to the past and the future is insignificant, and human existence is sole

# 7. Retrieval-Augmented Generation (RAG)

Large Language Models rely primarily on the knowledge learned during pretraining. However, they may lack information about specific domains or recent knowledge.

Retrieval-Augmented Generation (RAG) addresses this limitation by retrieving relevant external information before generating or ranking answers. This combines semantic search with language models to improve factual accuracy and reasoning.

## Building the Vector Database

The training questions are converted into dense sentence embeddings and stored as a searchable vector database. During inference, similar questions can be retrieved efficiently using cosine similarity.

In [14]:
train_embeddings = embedding_model.encode(
    train["prompt"].tolist(),
    convert_to_numpy=True,
    show_progress_bar=True
)

print(train_embeddings.shape)

Batches:   0%|          | 0/63 [00:00<?, ?it/s]

(2000, 384)


## Retrieving Relevant Context

For each query, we retrieve the most semantically similar questions from the vector database using cosine similarity. These retrieved examples provide additional context that can improve downstream reasoning and answer ranking.

In [15]:
sample_query = train.loc[0, "prompt"]

query_embedding = embedding_model.encode(
    sample_query,
    convert_to_numpy=True
)

similarities = cosine_similarity(
    [query_embedding],
    train_embeddings
).flatten()

top_k = 5

indices = similarities.argsort()[-top_k-1:-1][::-1]

retrieved = train.iloc[indices][["prompt", "answer"]]

retrieved

,prompt,answer
1823,pick the best possible answer: what is martin ...,B
3,select the most accurate option: what is marti...,B
921,select the most accurate option: what is marti...,B
1691,select the most accurate option: what is marti...,B
633,determine the correct option: what is martin h...,B


### Prompt Augmentation

The retrieved questions are combined with the original question to provide additional context before inference. This is the core idea behind Retrieval-Augmented Generation (RAG), where external knowledge is injected into the prompt instead of relying solely on the model's internal knowledge.

In [16]:
context = "\n\n".join(
    retrieved["prompt"].tolist()
)

augmented_prompt = f"""
Context:
{context}

Question:
{sample_query}
"""

print(augmented_prompt[:1000])


Context:
pick the best possible answer: what is martin heidegger's view on the relationship between time and human existence? among the listed options.

select the most accurate option: what is martin heidegger's view on the relationship between time and human existence? carefully.

select the most accurate option: what is martin heidegger's view on the relationship between time and human existence? carefully.

select the most accurate option: what is martin heidegger's view on the relationship between time and human existence? from the following choices.

determine the correct option: what is martin heidegger's view on the relationship between time and human existence? carefully.

Question:
pick the best possible answer: what is martin heidegger's view on the relationship between time and human existence? among the listed options.



# 8. Fine-tuning a Transformer

In here, we fine-tune a pretrained DeBERTa model for the multiple-choice question answering task using LoRA (Low-Rank Adaptation). Each question is paired with all five answer choices, allowing the transformer to compare the candidates simultaneously and learn which option is correct.

In [17]:
from datasets import Dataset

label2id = {label: idx for idx, label in enumerate(OPTIONS)}
id2label = {idx: label for label, idx in label2id.items()}

train_dataset = Dataset.from_pandas(
    train[
        ["prompt", "A", "B", "C", "D", "E", "answer"]
    ].rename(columns={"prompt": "question"})
)

train_dataset = train_dataset.map(
    lambda x: {
        "label": label2id[x["answer"]]
    }
)

train_dataset

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Dataset({
    features: ['question', 'A', 'B', 'C', 'D', 'E', 'answer', 'label'],
    num_rows: 2000
})

## Tokenizing the Multiple Choice Dataset

Each question is paired with every answer choice individually before tokenization. This produces five tokenized sequences for every question, allowing the transformer to compare all candidate answers simultaneously during training.

In [22]:
from transformers import AutoTokenizer

MODEL_NAME = "microsoft/deberta-v3-small"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


def preprocess_function(examples):
    first_sentences = []
    second_sentences = []

    for question, a, b, c, d, e in zip(
        examples["question"],
        examples["A"],
        examples["B"],
        examples["C"],
        examples["D"],
        examples["E"],
    ):
        choices = [a, b, c, d, e]

        first_sentences.extend([question] * 5)
        second_sentences.extend(choices)

    tokenized = tokenizer(
        first_sentences,
        second_sentences,
        truncation=True,
        max_length=256,
    )

    return {
        "input_ids": [
            tokenized["input_ids"][i:i + 5]
            for i in range(0, len(tokenized["input_ids"]), 5)
        ],
        "attention_mask": [
            tokenized["attention_mask"][i:i + 5]
            for i in range(0, len(tokenized["attention_mask"]), 5)
        ],
    }


tokenized_dataset = train_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=[
        "question",
        "A",
        "B",
        "C",
        "D",
        "E",
        "answer",
    ],
)

tokenized_dataset

config.json:   0%|          | 0.00/578 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Dataset({
    features: ['label', 'input_ids', 'attention_mask'],
    num_rows: 2000
})

## Loading DeBERTa and Configuring LoRA

A pretrained DeBERTa-v3 model is loaded for the multiple-choice task. Instead of updating all model parameters, LoRA (Low-Rank Adaptation) inserts lightweight trainable adapters into the attention layers, enabling efficient fine-tuning with significantly lower GPU memory requirements.

Weights & Biases (W&B) is initialized to track training metrics and compare experiments.

In [23]:
import os
import torch
import wandb

from kaggle_secrets import UserSecretsClient

from transformers import (
    AutoModelForMultipleChoice
)

from peft import (
    LoraConfig,
    TaskType,
    get_peft_model
)

user_secrets = UserSecretsClient()
os.environ["WANDB_API_KEY"] = user_secrets.get_secret("WandB_API_key")

wandb.login()

wandb.init(
    project="23f2004742-t22026",
    name="deberta-v3-small-lora"
)

model = AutoModelForMultipleChoice.from_pretrained(MODEL_NAME)

lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    bias="none",
    target_modules=[
        "query_proj",
        "value_proj"
    ]
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

wandb: WARNING Calling wandb.login() after wandb.init() has no effect.


pytorch_model.bin:   0%|          | 0.00/286M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2ForMultipleChoice LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
classifier.bias                         | MISSING    | 
pooler.dense.bias                       | MISSING    | 
pooler.dense.weight                     | MISSING    | 
classifier.weight               

trainable params: 148,225 || all params: 142,043,906 || trainable%: 0.1044


## Configuring the Trainer

The dataset is split into training and validation sets to evaluate the model during fine-tuning. A custom data collator dynamically pads each batch while preserving the five answer choices per question. The Hugging Face `Trainer` API is then configured to manage optimization, evaluation, checkpointing, and experiment tracking through Weights & Biases.

In [24]:
import numpy as np
from dataclasses import dataclass
from typing import Optional, Union

from sklearn.metrics import accuracy_score, f1_score

from transformers import Trainer, TrainingArguments

dataset = tokenized_dataset.train_test_split(
    test_size=0.2,
    seed=SEED
)

train_ds = dataset["train"]
valid_ds = dataset["test"]


@dataclass
class DataCollatorForMultipleChoice:
    tokenizer: AutoTokenizer
    padding: Union[bool, str] = True
    max_length: Optional[int] = None
    pad_to_multiple_of: Optional[int] = None

    def __call__(self, features):

        labels = [feature.pop("label") for feature in features]

        batch_size = len(features)
        num_choices = len(features[0]["input_ids"])

        flattened_features = [
            [
                {
                    "input_ids": feature["input_ids"][i],
                    "attention_mask": feature["attention_mask"][i],
                }
                for i in range(num_choices)
            ]
            for feature in features
        ]

        flattened_features = sum(flattened_features, [])

        batch = self.tokenizer.pad(
            flattened_features,
            padding=self.padding,
            max_length=self.max_length,
            pad_to_multiple_of=self.pad_to_multiple_of,
            return_tensors="pt",
        )

        batch = {
            k: v.view(batch_size, num_choices, -1)
            for k, v in batch.items()
        }

        batch["labels"] = torch.tensor(labels)

        return batch


data_collator = DataCollatorForMultipleChoice(tokenizer)


def compute_metrics(eval_pred):

    predictions, labels = eval_pred

    pred = np.argmax(predictions, axis=1)

    accuracy = accuracy_score(labels, pred)

    macro_f1 = f1_score(
        labels,
        pred,
        average="macro"
    )

    top3 = np.argsort(predictions, axis=1)[:, ::-1][:, :3]

    map3 = map_at_3(
        [id2label[i] for i in labels],
        [[id2label[j] for j in row] for row in top3]
    )

    return {
        "accuracy": accuracy,
        "macro_f1": macro_f1,
        "map@3": map3,
    }


training_args = TrainingArguments(
    output_dir="./mcq_model",

    learning_rate=2e-5,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    num_train_epochs=3,

    weight_decay=0.01,

    fp16=False,
    bf16=False,

    eval_strategy="epoch",
    save_strategy="epoch",

    logging_steps=20,

    load_best_model_at_end=True,

    metric_for_best_model="map@3",

    greater_is_better=True,

    report_to="wandb",

    remove_unused_columns=False,

    seed=SEED
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=valid_ds,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

model.safetensors:   0%|          | 0.00/286M [00:00<?, ?B/s]

In [25]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 1}.
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Map@3
1,3.212793,3.203125,0.295000,0.288653,0.482500
2,3.211621,3.203125,0.290000,0.284628,0.480417
3,3.200586,3.203125,0.292500,0.286663,0.482500


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


TrainOutput(global_step=300, training_loss=3.221360677083333, metrics={'train_runtime': 70.945, 'train_samples_per_second': 67.658, 'train_steps_per_second': 4.229, 'total_flos': 691688230401600.0, 'train_loss': 3.221360677083333, 'epoch': 3.0})